# Embedding Generation (Shared)This notebook generates and saves all embeddings (text, image, hybrid) once.Run this after `preprocessing.ipynb` and before any model notebook.

In [ ]:
import osimport torchimport numpy as npimport pandas as pdimport yamlfrom pathlib import Pathfrom PIL import Imagefrom tqdm.auto import tqdmfrom torch.utils.data import Dataset, DataLoaderfrom sentence_transformers import SentenceTransformerfrom transformers import CLIPProcessor, CLIPModel# Load Configwith open("config.yaml", 'r') as f:    config = yaml.safe_load(f)DATASET_DIR = config['paths']['dataset_dir']METADATA_PATH = os.path.join(DATASET_DIR, config['paths']['metadata_path'])IMAGES_ROOT = os.path.join(DATASET_DIR, config['paths']['images_root'])EMBEDDINGS_DIR = config['paths']['embeddings_dir']os.makedirs(EMBEDDINGS_DIR, exist_ok=True)TEXT_MODEL_ID = config['models']['text_model_id']IMAGE_MODEL_ID = config['models']['image_model_id']TEXT_BATCH_SIZE = config['pipeline']['text_batch_size']IMAGE_BATCH_SIZE = config['pipeline']['image_batch_size']DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"\u2699\uFE0F  Config Loaded")print(f"   - Device: {DEVICE}")print(f"   - Embeddings will be saved to: {EMBEDDINGS_DIR}")# Load metadatadf = pd.read_parquet(METADATA_PATH)print(f"   - Found {len(df):,} products.")

# 1. Text Embeddings (MPNet)

In [ ]:
TEXT_EMB_PATH = os.path.join(EMBEDDINGS_DIR, "text_embeddings.npy")if os.path.exists(TEXT_EMB_PATH):    print(f"\u2705 Text embeddings already exist: {TEXT_EMB_PATH}")    text_vectors = np.load(TEXT_EMB_PATH)else:    print(f"\U0001F680 Loading Text Model: {TEXT_MODEL_ID}...")    txt_model = SentenceTransformer(TEXT_MODEL_ID, device=DEVICE)        print("\u26A1 Generating Text Embeddings...")    text_vectors = txt_model.encode(        df['unified_text'].tolist(),        batch_size=TEXT_BATCH_SIZE,        show_progress_bar=True,        normalize_embeddings=True    )        np.save(TEXT_EMB_PATH, text_vectors)    print(f"   \u2705 Saved: {TEXT_EMB_PATH} - Shape: {text_vectors.shape}")    del txt_model    torch.cuda.empty_cache() if DEVICE == "cuda" else None

# 2. Image Embeddings (CLIP)

In [ ]:
IMAGE_EMB_PATH = os.path.join(EMBEDDINGS_DIR, "image_embeddings.npy")class TorobImageDataset(Dataset):    def __init__(self, dataframe, images_root):        self.df = dataframe        self.images_root = images_root            def __len__(self):        return len(self.df)        def __getitem__(self, idx):        row = self.df.iloc[idx]        img_name = os.path.basename(row['image_path'])        img_path = os.path.join(self.images_root, img_name)        try:            image = Image.open(img_path).convert("RGB")        except Exception:            image = Image.new('RGB', (224, 224), color='black')        return imageif os.path.exists(IMAGE_EMB_PATH):    print(f"\u2705 Image embeddings already exist: {IMAGE_EMB_PATH}")    image_vectors = np.load(IMAGE_EMB_PATH)else:    print(f"\U0001F680 Loading CLIP Model: {IMAGE_MODEL_ID}...")    img_model = CLIPModel.from_pretrained(IMAGE_MODEL_ID).to(DEVICE)    img_processor = CLIPProcessor.from_pretrained(IMAGE_MODEL_ID)    img_model.eval()        dataset = TorobImageDataset(df, IMAGES_ROOT)    loader = DataLoader(dataset, batch_size=IMAGE_BATCH_SIZE, shuffle=False, num_workers=2,                         collate_fn=lambda batch: batch)        all_embeddings = []    print("\u26A1 Generating Image Embeddings...")    with torch.no_grad():        for batch_images in tqdm(loader, desc="Encoding Images"):            inputs = img_processor(images=batch_images, return_tensors="pt", padding=True).to(DEVICE)            outputs = img_model.get_image_features(**inputs)            embeddings = outputs / outputs.norm(p=2, dim=-1, keepdim=True)            all_embeddings.append(embeddings.cpu().numpy())        image_vectors = np.vstack(all_embeddings)    np.save(IMAGE_EMB_PATH, image_vectors)    print(f"   \u2705 Saved: {IMAGE_EMB_PATH} - Shape: {image_vectors.shape}")    del img_model, img_processor    torch.cuda.empty_cache() if DEVICE == "cuda" else None

# 3. Hybrid Embeddings (Text + Image Concatenation)

In [ ]:
HYBRID_EMB_PATH = os.path.join(EMBEDDINGS_DIR, "hybrid_embeddings.npy")if os.path.exists(HYBRID_EMB_PATH):    print(f"\u2705 Hybrid embeddings already exist: {HYBRID_EMB_PATH}")    hybrid_vectors = np.load(HYBRID_EMB_PATH)else:    # Load text and image embeddings (they should exist by now)    text_vectors = np.load(TEXT_EMB_PATH)    image_vectors = np.load(IMAGE_EMB_PATH)        print("\U0001F517 Creating Hybrid Embeddings (concatenation)...")    hybrid_vectors = np.concatenate([text_vectors, image_vectors], axis=1)        np.save(HYBRID_EMB_PATH, hybrid_vectors)    print(f"   \u2705 Saved: {HYBRID_EMB_PATH} - Shape: {hybrid_vectors.shape}")print("\n\U0001F389 All embeddings generated and saved!")print(f"   - Text:   {TEXT_EMB_PATH}")print(f"   - Image:  {IMAGE_EMB_PATH}")print(f"   - Hybrid: {HYBRID_EMB_PATH}")